# 02 · Data Cleaning

This notebook documents the cleaning pipeline applied to the raw SMS Spam Collection before splitting and training.

## Cleaning Steps
1. Drop rows with null text or label
2. Strip whitespace and normalize labels to lowercase
3. Remove exact duplicate messages (keep first occurrence)
4. Validate that only `ham`/`spam` labels remain
5. Preserve security-relevant signals (URLs, phone numbers, currency symbols)

In [ ]:
import pandas as pd
from pathlib import Path

RAW_DIR = Path('../data/raw')
df = pd.read_csv(RAW_DIR / 'SMSSpamCollection', sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
original_count = len(df)
print(f'Raw messages: {original_count:,}')

In [ ]:
# Step 1: Drop nulls
df = df.dropna(subset=['text', 'label'])
print(f'After null removal: {len(df):,} (removed {original_count - len(df)})')

# Step 2: Normalize
df['text'] = df['text'].str.strip()
df['label'] = df['label'].str.strip().str.lower()

# Step 3: Remove duplicates
before_dedup = len(df)
df = df.drop_duplicates(subset=['text'], keep='first')
print(f'After deduplication: {len(df):,} (removed {before_dedup - len(df)} duplicates)')

# Step 4: Validate labels
df = df[df['label'].isin({'ham', 'spam'})]
df = df.reset_index(drop=True)
print(f'Final cleaned dataset: {len(df):,} messages')

## Duplicate Analysis

Understanding what was removed helps confirm we aren't discarding useful signal.

In [ ]:
# Reload to show what duplicates looked like
raw = pd.read_csv(RAW_DIR / 'SMSSpamCollection', sep='\t', header=None, names=['label', 'text'], encoding='latin-1')
raw['label'] = raw['label'].str.strip().str.lower()
dup_mask = raw.duplicated(subset=['text'], keep=False)
dups = raw[dup_mask].sort_values('text')

print(f'Total duplicate instances: {len(dups)} (from {dups["text"].nunique()} unique texts)')
print(f'\nDuplicate label distribution:')
print(dups['label'].value_counts())
print(f'\nSample duplicated messages:')
dups.head(6)

## Post-Cleaning Distribution

In [ ]:
print(f'Cleaned dataset shape: {df.shape}')
print(f'\nLabel distribution:')
print(df['label'].value_counts())
print(f'\nClass ratio: {df["label"].value_counts()["ham"] / df["label"].value_counts()["spam"]:.1f}:1 (ham:spam)')
print(f'Spam percentage: {(df["label"] == "spam").mean() * 100:.1f}%')

# Save cleaned dataset
out_dir = Path('../data/processed')
out_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(out_dir / 'dataset.csv', index=False)
print(f'\n✅ Cleaned data saved to {out_dir / "dataset.csv"}')